# FBI Crime Data — Data Cleaning

## 0. Import & Load Data

In [16]:
import pandas as pd
import numpy as np
raw = pd.read_excel('fbi.xlsx', header=None)

## 1. Exploratory Data Analysis

In [17]:
raw.shape

(200, 23)

In [18]:
# inspect raw double header rows
raw.head(5)

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
0,Area,Year,Population2,Violent crime3,NaN,Murder and \nnonnegligent \nmanslaughter,NaN,Rape\n(revised definition)4,NaN,Rape\n(legacy definition)5,...,Aggravated assault,NaN,Property crime,NaN,Burglary,NaN,Larceny-theft,NaN,Motor vehicle theft,NaN
1,NaN,NaN,NaN,,"Rate per \n100,000",,"Rate per \n100,000",,"Rate per \n100,000",,...,,"Rate per \n100,000",,"Rate per \n100,000",,"Rate per \n100,000",,"Rate per \n100,000",,"Rate per \n100,000"
2,"United States Total6, 7, 8, 9",2015,320896618,1234183,384.6,15883,4.9,126134,39.3,91261,...,764057,238.1,8024115,2500.5,1587564,494.7,5723488,1783.6,713063,222.2
3,NaN,2016,323127513,1283058,397.1,17250,5.3,130603,40.4,95730,...,803007,248.5,7919035,2450.7,1515096,468.9,5638455,1745,765484,236.9
4,NaN,Percent change,NaN,4,3.2,8.6,7.9,3.5,2.8,4.9,...,5.1,4.4,-1.3,-2,-4.6,-5.2,-1.5,-2.2,7.4,6.6


In [19]:
raw.tail(5)

,0,1,2,3,4,5,6,7,8,9,...,13,14,15,16,17,18,19,20,21,22
195,NaN,2016,7288000,22023,302.2,195,2.7,3077,42.2,2259,...,13100,179.7,254653,3494.1,49180,674.8,173187,2376.3,32286,443
196,NaN,Percent change,NaN,7.4,5.5,-10.6,-12.1,10.8,8.8,12.6,...,8.6,6.7,2.1,0.3,-3.8,-5.5,1,-0.7,19.7,17.6
197,Puerto Rico9,2015,3473181,7629,219.7,584,16.8,165,4.8,123,...,2810,80.9,37717,1085.9,9150,263.4,24493,705.2,4074,117.3
198,NaN,2016,3411307,7643,224,679,19.9,169,5,128,...,3594,105.4,35201,1031.9,8251,241.9,23163,679,3787,111
199,NaN,Percent change,,0.2,2,16.3,18.4,2.4,4.3,4.1,...,27.9,30.2,-6.7,-5,-9.8,-8.2,-5.4,-3.7,-7,-5.4


In [20]:
# all unique values in the Area column before cleaning
raw[0].dropna().unique()

array(['Area', 'United States Total6, 7, 8, 9', 'Northeast6',
       'New England6', '     Connecticut', '     Maine',
       '     Massachusetts', '     New Hampshire', '     Rhode Island6',
       '     Vermont', 'Middle Atlantic6', '     New Jersey',
       '     New York6', '     Pennsylvania', 'Midwest6',
       'East North Central6', '     Illinois', '     Indiana6',
       '     Michigan', '     Ohio6', '     Wisconsin6',
       'West North Central6', '     Iowa', '     Kansas',
       '     Minnesota', '     Missouri', '     Nebraska6',
       '     North Dakota', '     South Dakota6', 'South6, 7 ,8',
       'South Atlantic6, 7, 8 ', '     Delaware',
       '     District of Columbia7', '     Florida', '     Georgia8',
       '     Maryland', '     North Carolina6', '     South Carolina',
       '     Virginia', '     West Virginia', 'East South Central6',
       '     Alabama', '     Kentucky', '     Mississippi6',
       '     Tennessee', 'West South Central6', '     Arkansas

## 2. Data Cleaning

In [6]:
# skip the 2 header rows
df = raw.iloc[2:].copy()

cols = [
    'Area', 'Year', 'Population',
    'Violent_Crime_Count', 'Violent_Crime_Rate',
    'Murder_Count', 'Murder_Rate',
    'Rape_Revised_Count', 'Rape_Revised_Rate',
    'Rape_Legacy_Count', 'Rape_Legacy_Rate',
    'Robbery_Count', 'Robbery_Rate',
    'Aggravated_Assault_Count', 'Aggravated_Assault_Rate',
    'Property_Crime_Count', 'Property_Crime_Rate',
    'Burglary_Count', 'Burglary_Rate',
    'Larceny_Theft_Count', 'Larceny_Theft_Rate',
    'Motor_Vehicle_Theft_Count', 'Motor_Vehicle_Theft_Rate'
]
df.columns = cols

In [21]:
df['Area'] = df['Area'].ffill()

df['Area'] = df['Area'].astype(str).str.strip()
df['Area'] = df['Area'].str.replace(r'[\d,\s]+$', '', regex=True).str.strip()

df['Area'].unique()

array(['United States Total', 'Northeast', 'New England', 'Connecticut',
       'Maine', 'Massachusetts', 'New Hampshire', 'Rhode Island',
       'Vermont', 'Middle Atlantic', 'New Jersey', 'New York',
       'Pennsylvania', 'Midwest', 'East North Central', 'Illinois',
       'Indiana', 'Michigan', 'Ohio', 'Wisconsin', 'West North Central',
       'Iowa', 'Kansas', 'Minnesota', 'Missouri', 'Nebraska',
       'North Dakota', 'South Dakota', 'South', 'South Atlantic',
       'Delaware', 'District of Columbia', 'Florida', 'Georgia',
       'Maryland', 'North Carolina', 'South Carolina', 'Virginia',
       'West Virginia', 'East South Central', 'Alabama', 'Kentucky',
       'Mississippi', 'Tennessee', 'West South Central', 'Arkansas',
       'Louisiana', 'Oklahoma', 'Texas', 'West', 'Mountain', 'Arizona',
       'Colorado', 'Idaho', 'Montana', 'Nevada', 'New Mexico', 'Utah',
       'Wyoming', 'Pacific', 'Alaska', 'California', 'Hawaii', 'Oregon',
       'Washington', 'Puerto Rico'], dtype=

In [22]:
# replace special missing value encodings with NaN
df = df.replace([' ', '', '*'], np.nan)

In [23]:
# add Row_Type to distinguish yearly data from percent-change rows
df['Row_Type'] = df['Year'].apply(lambda y: 'Percent_Change' if y == 'Percent change' else 'Yearly')
df['Row_Type'].value_counts()

Row_Type
Yearly    198
Name: count, dtype: int64

In [10]:
# convert all numeric columns
numeric_cols = cols[2:]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Year: int for yearly rows, NaN for percent-change rows
df['Year'] = df['Year'].apply(lambda y: int(y) if y in [2015, 2016] else np.nan)

## 3. Validation

In [11]:
df.shape

(198, 24)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 2 to 199
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Area                       198 non-null    object 
 1   Year                       132 non-null    float64
 2   Population                 132 non-null    float64
 3   Violent_Crime_Count        198 non-null    float64
 4   Violent_Crime_Rate         198 non-null    float64
 5   Murder_Count               198 non-null    float64
 6   Murder_Rate                198 non-null    float64
 7   Rape_Revised_Count         198 non-null    float64
 8   Rape_Revised_Rate          197 non-null    float64
 9   Rape_Legacy_Count          198 non-null    float64
 10  Rape_Legacy_Rate           198 non-null    float64
 11  Robbery_Count              198 non-null    float64
 12  Robbery_Rate               197 non-null    float64
 13  Aggravated_Assault_Count   198 non-null    float64

In [13]:
df.isnull().sum()

Area                          0
Year                         66
Population                   66
Violent_Crime_Count           0
Violent_Crime_Rate            0
Murder_Count                  0
Murder_Rate                   0
Rape_Revised_Count            0
Rape_Revised_Rate             1
Rape_Legacy_Count             0
Rape_Legacy_Rate              0
Robbery_Count                 0
Robbery_Rate                  1
Aggravated_Assault_Count      0
Aggravated_Assault_Rate       1
Property_Crime_Count          0
Property_Crime_Rate           1
Burglary_Count                1
Burglary_Rate                 0
Larceny_Theft_Count           0
Larceny_Theft_Rate            0
Motor_Vehicle_Theft_Count     0
Motor_Vehicle_Theft_Rate      0
Row_Type                      0
dtype: int64

In [14]:
df.head(6)

,Area,Year,Population,Violent_Crime_Count,Violent_Crime_Rate,Murder_Count,Murder_Rate,Rape_Revised_Count,Rape_Revised_Rate,Rape_Legacy_Count,...,Aggravated_Assault_Rate,Property_Crime_Count,Property_Crime_Rate,Burglary_Count,Burglary_Rate,Larceny_Theft_Count,Larceny_Theft_Rate,Motor_Vehicle_Theft_Count,Motor_Vehicle_Theft_Rate,Row_Type
2,United States Total,2015.0,320896618.0,1234183.0,384.6,15883.0,4.9,126134.0,39.3,91261.0,...,238.1,8024115.0,2500.5,1587564.0,494.7,5723488.0,1783.6,713063.0,222.2,Yearly
3,United States Total,2016.0,323127513.0,1283058.0,397.1,17250.0,5.3,130603.0,40.4,95730.0,...,248.5,7919035.0,2450.7,1515096.0,468.9,5638455.0,1745.0,765484.0,236.9,Yearly
4,United States Total,NaN,NaN,4.0,3.2,8.6,7.9,3.5,2.8,4.9,...,4.4,-1.3,-2.0,-4.6,-5.2,-1.5,-2.2,7.4,6.6,Percent_Change
5,Northeast,2015.0,56184737.0,180474.0,321.2,1967.0,3.5,16415.0,29.2,11912.0,...,188.5,951940.0,1694.3,157910.0,281.1,737087.0,1311.9,56943.0,101.3,Yearly
6,Northeast,2016.0,56209510.0,178244.0,317.1,1955.0,3.5,16651.0,29.6,12219.0,...,189.7,909897.0,1618.8,142720.0,253.9,709721.0,1262.6,57456.0,102.2,Yearly
7,Northeast,NaN,NaN,-1.2,-1.3,-0.6,-0.7,1.4,1.4,2.6,...,0.6,-4.4,-4.5,-9.6,-9.7,-3.7,-3.8,0.9,0.9,Percent_Change


## 4. Save

In [15]:
df.to_csv('fbi_cleaned.csv', index=False)
print('Saved: fbi_cleaned.csv')

Saved: fbi_cleaned.csv
